In [1]:
# ============================================================
# PFE Pruning Experiments v7 — Fixed & Taylor-free
# Methods: 1a, 1b, 2, 3, 5, 6, 7  (Taylor/Method4 removed)
#          Movement pruning integrated as Method 8
# ============================================================

import torch, torchvision, time, os, json, copy, random
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from thop import profile
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
BASELINE    = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON = "__3__pruning_results_v7.json"
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY            = 0.50
CHANNEL_PRUNE_RATIO = 0.30

BASELINE_EPOCHS = 50

FINETUNE_EPOCHS = 10
FINETUNE_LR     = 1e-3

N_ROUNDS       = 3
_base_ft       = FINETUNE_EPOCHS // N_ROUNDS
ITER_FT_EPOCHS = [_base_ft + (1 if r == 0 else 0) for r in range(N_ROUNDS)]  # [4,3,3]
assert sum(ITER_FT_EPOCHS) == FINETUNE_EPOCHS

LTH_TRAIN_EPOCHS   = BASELINE_EPOCHS
LTH_RETRAIN_EPOCHS = BASELINE_EPOCHS

TIMING_WARMUP = 50
TIMING_REPS   = 500

print(f"Device           : {DEVICE}")
print(f"Sparsity target  : {SPARSITY*100:.0f}%")
print(f"Channel ratio    : {CHANNEL_PRUNE_RATIO*100:.0f}%")
print(f"Baseline epochs  : {BASELINE_EPOCHS}")
print(f"Fine-tune budget : {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}")
print(f"Iterative ft     : {ITER_FT_EPOCHS} per round  (sum={sum(ITER_FT_EPOCHS)})")
print(f"LTH train/retrain: {LTH_TRAIN_EPOCHS} + {LTH_RETRAIN_EPOCHS} epochs")

# ── MODEL FACTORY ─────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    m = models.resnet50(pretrained=False)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_baseline():
    m = build_model()
    m.load_state_dict(torch.load(BASELINE, map_location=DEVICE))
    return m.to(DEVICE)

# ── DATA ──────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set    = torchvision.datasets.CIFAR10('../data', True,  download=True,
                                            transform=transform_train)
test_set     = torchvision.datasets.CIFAR10('../data', False, download=True,
                                            transform=transform_test)
train_loader = torch.utils.data.DataLoader(
    train_set, BATCH_SIZE, shuffle=True,  num_workers=0,
    worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(
    test_set,  BATCH_SIZE, shuffle=False, num_workers=0)

# ── METRIC HELPERS ────────────────────────────────────────────
def evaluate_full(model):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    p, l = np.array(preds), np.array(labels)
    return dict(
        accuracy  = float(accuracy_score(l, p)),
        precision = float(precision_score(l, p, average='macro', zero_division=0)),
        recall    = float(recall_score(l, p, average='macro', zero_division=0)),
        f1        = float(f1_score(l, p, average='macro', zero_division=0)),
    )

def count_params(model):  return int(sum(p.numel() for p in model.parameters()))
def count_nonzero(model): return int(sum((p != 0).sum().item() for p in model.parameters()))

def model_size_mb(model, path="_tmp_.pth"):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return float(mb)

def theoretical_size_mb(model):
    """Nonzero float32 weights only — theoretical compressed floor."""
    nz = count_nonzero(model)
    return round((nz * 4) / 1e6, 4)

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return float(macs * 2 / 1e9)

def inference_ms_fn(model):
    """Single-sample latency with CUDA synchronization."""
    model.eval()
    dummy  = torch.randn(1, 3, 32, 32).to(DEVICE)
    is_gpu = DEVICE.type == "cuda"
    with torch.no_grad():
        for _ in range(TIMING_WARMUP):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(TIMING_REPS):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
    return float((time.perf_counter() - t0) / TIMING_REPS * 1000)

def collect_metrics(model, flops_note=None):
    """All metrics. inference_ms filled in the end-of-script timing pass."""
    m   = evaluate_full(model)
    tot = count_params(model)
    nz  = count_nonzero(model)
    m.update(dict(
        params_total         = tot,
        params_nonzero       = nz,
        sparsity_actual      = float(1 - nz / tot),
        size_mb              = model_size_mb(model),
        size_mb_theoretical  = theoretical_size_mb(model),
        flops_G              = compute_flops(model),
        inference_ms         = None,   # filled later in timing pass
    ))
    if flops_note:
        m["flops_note"] = flops_note
    return m

FLOPS_NOTE_MASK = (
    "Mask-based pruning — computation graph unchanged. "
    "Dense FLOPs identical to baseline. "
    "Effective FLOPs require sparse-kernel backend (e.g. DeepSparse)."
)

# ── TRAINING UTILITIES ────────────────────────────────────────
def train_model(model, epochs, lr=0.1, desc="train", frozen_mask=None):
    """SGD + CosineAnnealingLR — matches baseline optimizer exactly."""
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            if frozen_mask is not None:
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if name in frozen_mask:
                            p.mul_(frozen_mask[name])
        scheduler.step()
        if (ep + 1) % 10 == 0 or ep == 0:
            acc = evaluate_full(model)["accuracy"]
            print(f"    [{desc}] ep {ep+1:>3}/{epochs}  acc={acc:.4f}")
    return model

def fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
              desc="ft", frozen_mask=None):
    return train_model(model, epochs, lr=lr, desc=desc, frozen_mask=frozen_mask)

def get_prunable_layers(model):
    """(module, 'weight') pairs for all Conv2d and Linear."""
    return [(mod, 'weight') for mod in model.modules()
            if isinstance(mod, (nn.Conv2d, nn.Linear))]

def make_permanent(model):
    """Bake masks into weight tensors and remove reparametrization buffers."""
    for mod, _ in get_prunable_layers(model):
        try:  prune.remove(mod, 'weight')
        except ValueError: pass
    return model

# ── LTH MASK HELPER ──────────────────────────────────────────
def get_conv_linear_weight_keys(model):
    keys = []
    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            key = f"{name}.weight" if name else "weight"
            keys.append(key)
    assert len(keys) >= 50, (
        f"get_conv_linear_weight_keys found only {len(keys)} keys. "
        "Expected ≥50 for ResNet-50."
    )
    return keys

# ── CHECKPOINT SAVING ─────────────────────────────────────────
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# BUG FIX 1: save_model always takes the model explicitly as first arg
def save_model(model, key):
    path = os.path.join(SAVE_DIR, f"{key}.pth")
    torch.save(model.state_dict(), path)
    print(f"  ✓ Saved → {path}")

# ── MODEL REGISTRY ────────────────────────────────────────────
model_registry = {}
results        = {}

Device           : cuda
Sparsity target  : 50%
Channel ratio    : 30%
Baseline epochs  : 50
Fine-tune budget : 10 epochs @ lr=0.001
Iterative ft     : [4, 3, 3] per round  (sum=10)
LTH train/retrain: 50 + 50 epochs


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [2]:
# ══════════════════════════════════════════════════════════════
# METHOD 1a — UNSTRUCTURED (global L1, no fine-tune)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("1a. UNSTRUCTURED PRUNING  (global L1, no fine-tune)")
print("="*65)
# BUG FIX 2: use a local variable, never reuse bare `model` across methods
# to avoid accidentally passing a stale / wrong model to save_model or collect_metrics
model_1a = load_baseline()
prune.global_unstructured(
    get_prunable_layers(model_1a),
    pruning_method=prune.L1Unstructured,
    amount=SPARSITY
)
make_permanent(model_1a)
results["1a_unstructured_noft"] = collect_metrics(model_1a, FLOPS_NOTE_MASK)
results["1a_unstructured_noft"]["method_note"] = (
    "Global L1 unstructured, no fine-tuning. "
    "Paired with 7_one_shot (per-layer, no ft) → isolates threshold scope. "
    "Paired with 1b_unstructured_ft (same pruning, adds ft) → isolates ft value."
)
model_registry["1a_unstructured_noft"] = model_1a
save_model(model_1a, "1a_unstructured_noft")
print(f"  acc={results['1a_unstructured_noft']['accuracy']:.4f}  "
      f"sparsity={results['1a_unstructured_noft']['sparsity_actual']:.3f}")


1a. UNSTRUCTURED PRUNING  (global L1, no fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  ✓ Saved → saved_models\1a_unstructured_noft.pth
  acc=0.9320  sparsity=0.499


In [3]:
# ══════════════════════════════════════════════════════════════
# METHOD 1b — UNSTRUCTURED (global L1, WITH fine-tune)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("1b. UNSTRUCTURED PRUNING  (global L1, WITH fine-tune)")
print("="*65)
model_1b = load_baseline()
prune.global_unstructured(
    get_prunable_layers(model_1b),
    pruning_method=prune.L1Unstructured,
    amount=SPARSITY
)
model_1b = fine_tune(model_1b, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                     desc="1b-global-ft")
make_permanent(model_1b)
results["1b_unstructured_ft"] = collect_metrics(model_1b, FLOPS_NOTE_MASK)
results["1b_unstructured_ft"]["method_note"] = (
    f"Global L1 unstructured + {FINETUNE_EPOCHS}-epoch fine-tune @ lr={FINETUNE_LR}. "
    "Paired with 1a (isolates ft value) and 3_magnitude (isolates threshold scope)."
)
model_registry["1b_unstructured_ft"] = model_1b
save_model(model_1b, "1b_unstructured_ft")
print(f"  acc={results['1b_unstructured_ft']['accuracy']:.4f}  "
      f"sparsity={results['1b_unstructured_ft']['sparsity_actual']:.3f}")


1b. UNSTRUCTURED PRUNING  (global L1, WITH fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [1b-global-ft] ep   1/10  acc=0.9302
    [1b-global-ft] ep  10/10  acc=0.9322
  ✓ Saved → saved_models\1b_unstructured_ft.pth
  acc=0.9322  sparsity=0.499


In [4]:
# ══════════════════════════════════════════════════════════════
# METHOD 2 — STRUCTURED PRUNING (channel removal, Bottleneck-safe)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("2. STRUCTURED PRUNING  (channel removal + fine-tune)")
print("="*65)

def _prune_conv_out(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(conv.in_channels, n, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data[kept_idx].clone()
    return new

def _prune_conv_in(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(n, conv.out_channels, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[:, kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data.clone()
    return new

def _prune_bn(bn, kept_idx):
    n   = len(kept_idx)
    new = nn.BatchNorm2d(n, eps=bn.eps, momentum=bn.momentum,
                          affine=bn.affine,
                          track_running_stats=bn.track_running_stats)
    if bn.affine:
        new.weight.data = bn.weight.data[kept_idx].clone()
        new.bias.data   = bn.bias.data[kept_idx].clone()
    if bn.track_running_stats:
        new.running_mean        = bn.running_mean[kept_idx].clone()
        new.running_var         = bn.running_var[kept_idx].clone()
        new.num_batches_tracked = bn.num_batches_tracked.clone()
    return new

def prune_resnet50_structured(model, ratio):
    model = copy.deepcopy(model)
    for stage in ['layer1', 'layer2', 'layer3', 'layer4']:
        for block in getattr(model, stage):
            conv1  = block.conv1
            bn1    = block.bn1
            conv2  = block.conv2
            scores = conv1.weight.data.view(conv1.out_channels, -1).norm(p=2, dim=1)
            n_keep = max(1, int(conv1.out_channels * (1 - ratio)))
            kept   = scores.argsort(descending=True)[:n_keep].sort().values
            block.conv1 = _prune_conv_out(conv1, kept)
            block.bn1   = _prune_bn(bn1, kept)
            block.conv2 = _prune_conv_in(conv2, kept)
            assert block.conv1.out_channels == block.conv2.in_channels, (
                f"Shape mismatch in {stage}: "
                f"conv1.out={block.conv1.out_channels} "
                f"conv2.in={block.conv2.in_channels}"
            )
    return model

model_struct = prune_resnet50_structured(
    load_baseline(), ratio=CHANNEL_PRUNE_RATIO).to(DEVICE)
model_struct = fine_tune(model_struct, epochs=FINETUNE_EPOCHS,
                          lr=FINETUNE_LR, desc="2-structured-ft")
results["2_structured"] = collect_metrics(model_struct)
results["2_structured"]["flops_note"] = (
    f"TRUE structured pruning: {CHANNEL_PRUNE_RATIO*100:.0f}% of internal "
    "Bottleneck channels physically removed (conv1+bn1+conv2 per block). "
    "conv3, bn3, downsample untouched — residual shapes guaranteed. "
    "GFLOPs genuinely reduced; measured by thop on rebuilt model."
)
model_registry["2_structured"] = model_struct
# BUG FIX 1: was save_model(model, "2_structured") — bare `model` was undefined/stale
save_model(model_struct, "2_structured")
print(f"  acc={results['2_structured']['accuracy']:.4f}  "
      f"flops={results['2_structured']['flops_G']:.3f} GFLOPs  "
      f"params={results['2_structured']['params_total']:,}")


2. STRUCTURED PRUNING  (channel removal + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-ft] ep   1/10  acc=0.9313
    [2-structured-ft] ep  10/10  acc=0.9322
  ✓ Saved → saved_models\2_structured.pth
  acc=0.9322  flops=2.069 GFLOPs  params=18,807,394


In [5]:
# ══════════════════════════════════════════════════════════════
# METHOD 3 — MAGNITUDE PRUNING (per-layer L1 + fine-tune)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("3. MAGNITUDE PRUNING  (per-layer L1 + fine-tune)")
print("="*65)
model_3 = load_baseline()
for mod, name in get_prunable_layers(model_3):
    prune.l1_unstructured(mod, name=name, amount=SPARSITY)
model_3 = fine_tune(model_3, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                    desc="3-magnitude-ft")
make_permanent(model_3)
results["3_magnitude"] = collect_metrics(model_3, FLOPS_NOTE_MASK)
results["3_magnitude"]["method_note"] = (
    f"Per-layer L1 magnitude pruning + {FINETUNE_EPOCHS}-epoch fine-tune "
    f"@ lr={FINETUNE_LR}. Han et al. 2015. Central reference method."
)
model_registry["3_magnitude"] = model_3
save_model(model_3, "3_magnitude")
print(f"  acc={results['3_magnitude']['accuracy']:.4f}  "
      f"sparsity={results['3_magnitude']['sparsity_actual']:.3f}")


3. MAGNITUDE PRUNING  (per-layer L1 + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [3-magnitude-ft] ep   1/10  acc=0.9311
    [3-magnitude-ft] ep  10/10  acc=0.9323
  ✓ Saved → saved_models\3_magnitude.pth
  acc=0.9323  sparsity=0.499


In [6]:
# ══════════════════════════════════════════════════════════════
# METHOD 5 — LOTTERY TICKET HYPOTHESIS
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("5. LOTTERY TICKET HYPOTHESIS  (conv+linear, budget-matched to baseline)")
print("="*65)

torch.manual_seed(SEED)
lth_model = build_model().to(DEVICE)
theta_0   = copy.deepcopy(lth_model.state_dict())

print(f"  [LTH] Training θ₀ → θ_T  ({LTH_TRAIN_EPOCHS} epochs, matches baseline) ...")
lth_model = train_model(lth_model, epochs=LTH_TRAIN_EPOCHS, lr=0.1, desc="LTH-train")
theta_T   = copy.deepcopy(lth_model.state_dict())

maskable_keys = get_conv_linear_weight_keys(lth_model)
all_vals  = torch.cat([theta_T[k].abs().view(-1) for k in maskable_keys])
threshold = torch.topk(
    all_vals, int(all_vals.numel() * SPARSITY), largest=False
).values.max()
# BUG FIX 3: `mask` must be on DEVICE so frozen_mask works correctly in train_model
mask = {k: (theta_T[k].abs() > threshold).float().to(DEVICE) for k in maskable_keys}
print(f"  [LTH] Mask built over {len(mask)} tensors (conv+linear only)")

ticket_state = {}
for k, v in theta_0.items():
    ticket_state[k] = (v.to(DEVICE) * mask[k]) if k in mask else v.to(DEVICE)
ticket_model = build_model().to(DEVICE)
ticket_model.load_state_dict(ticket_state)

print(f"  [LTH] Retraining winning ticket ({LTH_RETRAIN_EPOCHS} epochs) ...")
ticket_model = train_model(ticket_model, epochs=LTH_RETRAIN_EPOCHS,
                            lr=0.1, desc="LTH-retrain", frozen_mask=mask)

results["5_lottery_ticket"] = collect_metrics(ticket_model, FLOPS_NOTE_MASK)
results["5_lottery_ticket"]["lth_note"] = (
    f"LTH — Frankle & Carlin 2019. θ₀ @ SEED={SEED}. "
    f"θ_T: {LTH_TRAIN_EPOCHS}-epoch in-script training (lr=0.1, SGD, ColorJitter). "
    f"Global magnitude mask @ {SPARSITY*100:.0f}% on Conv2d+Linear weights only. "
    f"Ticket retrained {LTH_RETRAIN_EPOCHS} epochs with frozen mask."
)
model_registry["5_lottery_ticket"] = ticket_model
save_model(ticket_model, "5_lottery_ticket")
print(f"  acc={results['5_lottery_ticket']['accuracy']:.4f}  "
      f"sparsity={results['5_lottery_ticket']['sparsity_actual']:.3f}")


5. LOTTERY TICKET HYPOTHESIS  (conv+linear, budget-matched to baseline)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  [LTH] Training θ₀ → θ_T  (50 epochs, matches baseline) ...
    [LTH-train] ep   1/50  acc=0.1988
    [LTH-train] ep  10/50  acc=0.6690
    [LTH-train] ep  20/50  acc=0.7918
    [LTH-train] ep  30/50  acc=0.8664
    [LTH-train] ep  40/50  acc=0.9161
    [LTH-train] ep  50/50  acc=0.9362
  [LTH] Mask built over 54 tensors (conv+linear only)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  [LTH] Retraining winning ticket (50 epochs) ...
    [LTH-retrain] ep   1/50  acc=0.2316
    [LTH-retrain] ep  10/50  acc=0.6846
    [LTH-retrain] ep  20/50  acc=0.7775
    [LTH-retrain] ep  30/50  acc=0.8595
    [LTH-retrain] ep  40/50  acc=0.9108
    [LTH-retrain] ep  50/50  acc=0.9337
  ✓ Saved → saved_models\5_lottery_ticket.pth
  acc=0.9337  sparsity=0.499


In [7]:
# ══════════════════════════════════════════════════════════════
# METHOD 6 — ITERATIVE PRUNING (N rounds, per-layer L1)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"6. ITERATIVE PRUNING  ({N_ROUNDS} rounds, per-layer L1, total ft={FINETUNE_EPOCHS} epochs)")
print("="*65)

S_ROUND = 1 - (1 - SPARSITY) ** (1 / N_ROUNDS)
model_6 = load_baseline()

for r in range(N_ROUNDS):
    ft_ep = ITER_FT_EPOCHS[r]
    for mod, name in get_prunable_layers(model_6):
        prune.l1_unstructured(mod, name=name, amount=S_ROUND)
    model_6 = fine_tune(model_6, epochs=ft_ep, lr=FINETUNE_LR,
                        desc=f"6-iter-r{r+1}(ft={ft_ep}ep)")
    total  = sum(p.numel() for p in model_6.parameters())
    zeroed = sum(
        (getattr(mod, 'weight_mask') == 0).sum().item()
        for mod in model_6.modules() if hasattr(mod, 'weight_mask')
    )
    print(f"  Round {r+1}/{N_ROUNDS}  mask-sparsity={zeroed/total:.3f}  "
          f"ft_epochs={ft_ep}  lr={FINETUNE_LR}")

# BUG FIX 4: make_permanent OUTSIDE the loop — masks AND-accumulate correctly
make_permanent(model_6)
nz  = count_nonzero(model_6)
tot = count_params(model_6)
print(f"  Final weight sparsity: {1-nz/tot:.3f}  (target={SPARSITY})")

results["6_iterative"] = collect_metrics(model_6, FLOPS_NOTE_MASK)
results["6_iterative"]["method_note"] = (
    f"{N_ROUNDS}-round per-layer L1 iterative pruning. "
    f"Per-round sparsity: {S_ROUND:.4f} → compounds to {SPARSITY*100:.0f}%. "
    f"Fine-tune epochs: {ITER_FT_EPOCHS} per round (total={sum(ITER_FT_EPOCHS)}). "
    f"Constant LR={FINETUNE_LR} across all rounds. "
    "make_permanent() called once after all rounds."
)
model_registry["6_iterative"] = model_6
save_model(model_6, "6_iterative")
print(f"  acc={results['6_iterative']['accuracy']:.4f}  "
      f"sparsity={results['6_iterative']['sparsity_actual']:.3f}")


6. ITERATIVE PRUNING  (3 rounds, per-layer L1, total ft=10 epochs)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [6-iter-r1(ft=4ep)] ep   1/4  acc=0.9311
  Round 1/3  mask-sparsity=0.206  ft_epochs=4  lr=0.001
    [6-iter-r2(ft=3ep)] ep   1/3  acc=0.9311
  Round 2/3  mask-sparsity=0.369  ft_epochs=3  lr=0.001
    [6-iter-r3(ft=3ep)] ep   1/3  acc=0.9308
  Round 3/3  mask-sparsity=0.499  ft_epochs=3  lr=0.001
  Final weight sparsity: 0.499  (target=0.5)
  ✓ Saved → saved_models\6_iterative.pth
  acc=0.9315  sparsity=0.499


In [8]:
# ══════════════════════════════════════════════════════════════
# METHOD 7 — ONE-SHOT PRUNING (per-layer L1, no fine-tune)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("7. ONE-SHOT PRUNING  (per-layer L1, zero fine-tune)")
print("="*65)
model_7 = load_baseline()
for mod, name in get_prunable_layers(model_7):
    prune.l1_unstructured(mod, name=name, amount=SPARSITY)
make_permanent(model_7)
results["7_one_shot"] = collect_metrics(model_7, FLOPS_NOTE_MASK)
results["7_one_shot"]["method_note"] = (
    f"Per-layer L1, {SPARSITY*100:.0f}% sparsity, zero fine-tuning. "
    "Paired with 1a (global threshold, no ft) → isolates threshold scope. "
    "Paired with 3 (same criterion, ft=Nep) → isolates fine-tuning value."
)
model_registry["7_one_shot"] = model_7
save_model(model_7, "7_one_shot")
print(f"  acc={results['7_one_shot']['accuracy']:.4f}  "
      f"sparsity={results['7_one_shot']['sparsity_actual']:.3f}")


7. ONE-SHOT PRUNING  (per-layer L1, zero fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  ✓ Saved → saved_models\7_one_shot.pth
  acc=0.9318  sparsity=0.499


In [9]:
# ══════════════════════════════════════════════════════════════
# METHOD 8 — MOVEMENT PRUNING (Sanh et al. 2020)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("8. MOVEMENT PRUNING  (Sanh et al. 2020)")
print("="*65)

model_8 = load_baseline()

# Snapshot W0 before fine-tuning
w_before = {name: p.detach().cpu().clone()
            for name, p in model_8.named_parameters() if p.requires_grad}

# Fine-tune (no mask yet — movement is measured AFTER training)
model_8 = fine_tune(model_8, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                    desc="movement-ft")

# Snapshot W1 after fine-tuning
w_after = {name: p.detach().cpu().clone()
           for name, p in model_8.named_parameters() if p.requires_grad}

# Score = sign(W0) * (W1 - W0): weights moving away from zero score high
scores_mv  = {}
all_scores = []
for name in w_before:
    delta        = w_after[name] - w_before[name]
    score        = torch.sign(w_before[name]) * delta
    scores_mv[name] = score
    all_scores.append(score.view(-1))
all_scores = torch.cat(all_scores)

# Global threshold — prune lowest-scoring SPARSITY fraction
k         = max(1, int(all_scores.numel() * SPARSITY))
# BUG FIX 5: kthvalue on absolute values so near-zero negatives aren't spared
threshold = torch.kthvalue(all_scores.abs(), k).values.item()

with torch.no_grad():
    for name, p in model_8.named_parameters():
        if name in scores_mv:
            mask_mv = (scores_mv[name].abs().to(p.device) > threshold).float()
            p.mul_(mask_mv)

results["8_movement_pruning"] = collect_metrics(model_8, FLOPS_NOTE_MASK)
results["8_movement_pruning"]["method_note"] = (
    "True movement pruning (Sanh et al. 2020). "
    "Score = sign(W0) * (W1 - W0). "
    "Weights moving toward zero are pruned globally. "
    "Mask applied post-training via p.mul_() — no live hook (movement requires ft first)."
)
model_registry["8_movement_pruning"] = model_8
save_model(model_8, "8_movement_pruning")
print(f"  acc={results['8_movement_pruning']['accuracy']:.4f}  "
      f"sparsity={results['8_movement_pruning']['sparsity_actual']:.3f}")


8. MOVEMENT PRUNING  (Sanh et al. 2020)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [movement-ft] ep   1/10  acc=0.9312
    [movement-ft] ep  10/10  acc=0.9317
  ✓ Saved → saved_models\8_movement_pruning.pth
  acc=0.9188  sparsity=0.500


In [10]:
# ══════════════════════════════════════════════════════════════
# END-OF-SCRIPT TIMING PASS
# All models timed consecutively under identical conditions.
# ══════════════════════════════════════════════════════════════
TIMING_ORDER = [
    "1a_unstructured_noft",
    "1b_unstructured_ft",
    "2_structured",
    "3_magnitude",
    "5_lottery_ticket",
    "6_iterative",
    "7_one_shot",
    "8_movement_pruning",
]

print("\n" + "="*65)
print(f"TIMING PASS  (warmup={TIMING_WARMUP}, reps={TIMING_REPS}, cuda_sync=True)")
print("="*65)
for key in TIMING_ORDER:
    ms = inference_ms_fn(model_registry[key])
    results[key]["inference_ms"] = ms
    print(f"  {key:<30}  {ms:.3f} ms")

# ── SAVE JSON ─────────────────────────────────────────────────
output = {
    "_meta": {
        "framework"             : "PFE Pruning Experiments v7 (Taylor-free)",
        "baseline_pth"          : BASELINE,
        "sparsity_target"       : SPARSITY,
        "channel_prune_ratio"   : CHANNEL_PRUNE_RATIO,
        "device"                : str(DEVICE),
        "seed"                  : SEED,
        "baseline_epochs"       : BASELINE_EPOCHS,
        "finetune_epochs"       : FINETUNE_EPOCHS,
        "finetune_lr"           : FINETUNE_LR,
        "iterative_ft_per_round": ITER_FT_EPOCHS,
        "lth_train_epochs"      : LTH_TRAIN_EPOCHS,
        "lth_retrain_epochs"    : LTH_RETRAIN_EPOCHS,
        "timing_config": {
            "warmup_reps": TIMING_WARMUP,
            "timed_reps" : TIMING_REPS,
            "cuda_sync"  : True,
        },
        "taxonomy": {
            "mask_based"           : ["1a_unstructured_noft", "1b_unstructured_ft",
                                      "3_magnitude", "6_iterative",
                                      "7_one_shot",  "8_movement_pruning"],
            "architecture_changing": ["2_structured"],
            "rewind_based"         : ["5_lottery_ticket"],
        },
        "paired_comparisons": {
            "threshold_scope_no_ft"  : "1a vs 7  → global vs per-layer (both: no ft)",
            "threshold_scope_with_ft": "1b vs 3  → global vs per-layer (both: ft=Nep)",
            "fine_tune_value"        : "7  vs 3  → no ft vs ft=Nep (both: per-layer L1)",
            "pruning_schedule"       : "3  vs 6  → one-shot vs iterative (both: per-layer L1)",
            "ft_value_global"        : "1a vs 1b → ft effect on global-pruned model",
            "criterion_movement"     : "3  vs 8  → L1-magnitude vs movement score",
        },
    },
    "results": results
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)

# ── SUMMARY TABLE ─────────────────────────────────────────────
LABELS = {
    "1a_unstructured_noft" : "1a. Unstructured (global, no-ft)",
    "1b_unstructured_ft"   : "1b. Unstructured (global, +ft)",
    "2_structured"         : "2.  Structured (channel)",
    "3_magnitude"          : "3.  Magnitude (per-layer, +ft)",
    "5_lottery_ticket"     : "5.  LTH",
    "6_iterative"          : "6.  Iterative (3-round, +ft)",
    "7_one_shot"           : "7.  One-Shot (per-layer, no-ft)",
    "8_movement_pruning"   : "8.  Movement Pruning",
}
TYPES = {
    "1a_unstructured_noft" : "mask",
    "1b_unstructured_ft"   : "mask",
    "2_structured"         : "arch",
    "3_magnitude"          : "mask",
    "5_lottery_ticket"     : "rewind",
    "6_iterative"          : "mask",
    "7_one_shot"           : "mask",
    "8_movement_pruning"   : "mask",
}

print("\n" + "="*90)
print(f"ALL DONE — {OUTPUT_JSON}")
print("="*90)
hdr = (f"{'Method':<38} {'Type':<7} {'Acc':>7} {'F1':>7} "
       f"{'Spar':>6} {'MB':>7} {'Theo MB':>8} {'GFLOPs':>8} {'ms':>8}")
print("\n" + hdr)
print("-" * len(hdr))
for k in TIMING_ORDER:
    m  = results[k]
    sp = m.get('sparsity_actual', 0.0)
    ms = m.get('inference_ms') or 0.0
    tb = m.get('size_mb_theoretical', 0.0)
    print(f"{LABELS.get(k,k):<38} {TYPES.get(k,'?'):<7} "
          f"{m['accuracy']:>7.4f} {m['f1']:>7.4f} {sp:>6.3f} "
          f"{m['size_mb']:>7.2f} {tb:>8.2f} {m['flops_G']:>8.3f} {ms:>8.3f}")

print()
print("  Type: mask=weight-masked  arch=physically-rebuilt  rewind=LTH")
print()
print("  Paired comparisons:")
print("  1a vs 7  → threshold scope    (global vs per-layer,  both no-ft)")
print("  1b vs 3  → threshold scope    (global vs per-layer,  both ft=Nep)")
print("  7  vs 3  → fine-tune value    (no-ft vs ft=Nep,      both per-layer L1)")
print("  3  vs 6  → pruning schedule   (one-shot vs iterative, both per-layer L1)")
print("  1a vs 1b → ft value (global)  (no-ft vs ft=Nep,      both global L1)")
print("  3  vs 8  → criterion          (L1-magnitude vs movement score)")
print(f"\n  Saved → {OUTPUT_JSON}")

# ── VERIFICATION BLOCK ────────────────────────────────────────
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)
ok = True

# V1: Fine-tuning helps (Method 3 > Method 7)
acc3 = results["3_magnitude"]["accuracy"]
acc7 = results["7_one_shot"]["accuracy"]
v1   = acc3 > acc7
print(f"\nV1  Method 3 (ft) vs 7 (no-ft): {acc3:.4f} vs {acc7:.4f}")
print(f"    → {'PASS ✓' if v1 else 'FAIL — check Method 3 fine-tune'}")
ok = ok and v1

# V2: Iterative total ft = FINETUNE_EPOCHS
total_iter_ft = sum(ITER_FT_EPOCHS)
v2 = total_iter_ft == FINETUNE_EPOCHS
print(f"\nV2  Iterative total ft epochs: {total_iter_ft}  (target={FINETUNE_EPOCHS})")
print(f"    Per-round: {ITER_FT_EPOCHS}")
print(f"    → {'PASS ✓' if v2 else 'FAIL'}")
ok = ok and v2

# V3: Iterative sparsity close to target
sp6 = results["6_iterative"]["sparsity_actual"]
v3  = abs(sp6 - SPARSITY) < 0.05
print(f"\nV3  Iterative sparsity: actual={sp6:.3f}  target={SPARSITY:.2f}")
print(f"    → {'PASS ✓' if v3 else f'FAIL — delta={abs(sp6-SPARSITY):.3f}'}")
ok = ok and v3

# V4: LTH mask covers only conv+linear (no BN keys)
# BUG FIX 6: `mask` is defined in the LTH block above — valid here
bn_in_mask = [k for k in mask if 'bn' in k or 'downsample.1' in k]
v4 = len(bn_in_mask) == 0
print(f"\nV4  LTH mask keys: {len(mask)} total, BN keys: {len(bn_in_mask)}")
print(f"    → {'PASS ✓ (conv+linear only)' if v4 else f'FAIL — BN keys: {bn_in_mask[:3]}'}")
ok = ok and v4

# V5: LTH mask key count
v5 = len(mask) >= 50
print(f"\nV5  LTH maskable key count: {len(mask)}  (expected ≥50 for ResNet-50)")
print(f"    → {'PASS ✓' if v5 else 'FAIL'}")
ok = ok and v5

# V6: LTH training epoch count matches baseline
v6 = LTH_TRAIN_EPOCHS == BASELINE_EPOCHS
print(f"\nV6  LTH_TRAIN_EPOCHS ({LTH_TRAIN_EPOCHS}) == BASELINE_EPOCHS ({BASELINE_EPOCHS})")
print(f"    → {'PASS ✓' if v6 else 'FAIL'}")
ok = ok and v6

# V7: ColorJitter in transform_train
v7 = 'ColorJitter' in str(transform_train)
print(f"\nV7  ColorJitter in transform_train: {v7}")
print(f"    → {'PASS ✓' if v7 else 'FAIL'}")
ok = ok and v7

# V8: Structured FLOPs < unstructured FLOPs (genuine reduction)
flops_struct = results["2_structured"]["flops_G"]
flops_mask   = results["1a_unstructured_noft"]["flops_G"]
v8 = flops_struct < flops_mask
print(f"\nV8  Structured GFLOPs {flops_struct:.3f} < mask baseline {flops_mask:.3f}")
print(f"    → {'PASS ✓' if v8 else 'FAIL'}")
ok = ok and v8

# V9: Timing is populated for all methods
v9 = all(results[k].get("inference_ms") is not None and results[k]["inference_ms"] > 0
         for k in TIMING_ORDER)
print(f"\nV9  inference_ms populated for all {len(TIMING_ORDER)} methods")
print(f"    → {'PASS ✓' if v9 else 'FAIL — some methods have None/zero inference_ms'}")
ok = ok and v9

# V10: Theoretical size ≤ actual size for all methods
v10 = all(results[k].get("size_mb_theoretical", 0) <= results[k]["size_mb"]
          for k in TIMING_ORDER)
print(f"\nV10 Theoretical size ≤ actual size for all methods")
print(f"    → {'PASS ✓' if v10 else 'FAIL — theoretical exceeds actual somewhere'}")
ok = ok and v10

# V11: No duplicate methods (by accuracy + nonzero params)
def model_hash(key):
    return f"{results[key]['accuracy']:.6f}_{results[key]['params_nonzero']}"
hashes = {k: model_hash(k) for k in TIMING_ORDER}
seen, v11 = {}, True
for k, h in hashes.items():
    if h in seen:
        print(f"\nV11 ⚠ DUPLICATE: {k} and {seen[h]}")
        v11 = False
    seen[h] = k
if v11:
    print(f"\nV11 No duplicate methods detected ✓")
ok = ok and v11

# V12: Movement sparsity close to target
sp8 = results["8_movement_pruning"]["sparsity_actual"]
v12 = abs(sp8 - SPARSITY) < 0.05
print(f"\nV12 Movement sparsity: actual={sp8:.3f}  target={SPARSITY:.2f}")
print(f"    → {'PASS ✓' if v12 else f'FAIL — delta={abs(sp8-SPARSITY):.3f}'}")
ok = ok and v12

print("\n" + "="*80)
print(f"OVERALL: {'ALL CHECKS PASSED ✓' if ok else 'SOME CHECKS FAILED — review above'}")
print("="*80)


TIMING PASS  (warmup=50, reps=500, cuda_sync=True)
  1a_unstructured_noft            18.700 ms
  1b_unstructured_ft              19.206 ms
  2_structured                    17.506 ms
  3_magnitude                     19.766 ms
  5_lottery_ticket                17.987 ms
  6_iterative                     19.022 ms
  7_one_shot                      17.914 ms
  8_movement_pruning              18.027 ms

ALL DONE — __3__pruning_results_v7.json

Method                                 Type        Acc      F1   Spar      MB  Theo MB   GFLOPs       ms
--------------------------------------------------------------------------------------------------------
1a. Unstructured (global, no-ft)       mask     0.9320  0.9319  0.499   94.38    47.15    2.623   18.700
1b. Unstructured (global, +ft)         mask     0.9322  0.9322  0.499   94.38    47.15    2.623   19.206
2.  Structured (channel)               arch     0.9322  0.9320  0.000   75.52    75.23    2.069   17.506
3.  Magnitude (per-layer, +ft